# Python 进阶练习册

本 Notebook 包含 **10 个主题、30+ 道练习题**，覆盖 Python 进阶知识。

### 使用方法

1. 阅读每道题的说明
2. 在标有 `# ✏️ 在此写代码` 的单元格中编写你的答案
3. 运行下方的 **✅ 验证** 单元格，系统会自动判断你的输出是否正确
4. 如果卡住了，可以展开 **💡 提示** 查看线索

### 主题一览

| # | 主题 | 难度 |
|---|------|------|
| 1 | 推导式与生成器 | ⭐⭐ |
| 2 | 装饰器 | ⭐⭐⭐ |
| 3 | 闭包与作用域 | ⭐⭐⭐ |
| 4 | 迭代器协议与 `yield` | ⭐⭐⭐ |
| 5 | 魔术方法与运算符重载 | ⭐⭐⭐ |
| 6 | 上下文管理器 | ⭐⭐⭐ |
| 7 | `*args` / `**kwargs` 与解包 | ⭐⭐ |
| 8 | 函数式编程 | ⭐⭐⭐ |
| 9 | Dataclass 与类型提示 | ⭐⭐ |
| 10 | 异步编程 async/await | ⭐⭐⭐⭐ |

## 初始化：运行验证工具

**请先运行下面的单元格**，它定义了用来检查你答案的辅助函数。

In [1]:
import traceback, inspect, types, sys, io, asyncio

class _Checker:
    def __init__(self):
        self._score = {}

    def check(self, name, expr, expected, *, compare=None, show_expected=True):
        """验证一个表达式的值是否符合预期"""
        try:
            actual = expr() if callable(expr) else expr
            ok = compare(actual, expected) if compare else (actual == expected)
            if ok:
                print(f"  ✅ {name}")
                self._score[name] = True
            else:
                hint = f"  期望: {expected!r}" if show_expected else ""
                print(f"  ❌ {name} — 你的结果: {actual!r}{hint}")
                self._score[name] = False
        except Exception as e:
            print(f"  ❌ {name} — 运行出错: {e}")
            self._score[name] = False

    def check_output(self, name, func, expected_contains, *args, **kwargs):
        """捕获 print 输出并检查是否包含期望文本"""
        buf = io.StringIO()
        old = sys.stdout
        try:
            sys.stdout = buf
            func(*args, **kwargs)
        finally:
            sys.stdout = old
        output = buf.getvalue()
        if all(s in output for s in expected_contains):
            print(f"  ✅ {name}")
            self._score[name] = True
        else:
            print(f"  ❌ {name} — 输出不符合预期")
            print(f"     你的输出: {output.strip()!r}")
            self._score[name] = False

    def check_type(self, name, obj, expected_type):
        """检查对象类型"""
        if isinstance(obj, expected_type):
            print(f"  ✅ {name}")
            self._score[name] = True
        else:
            print(f"  ❌ {name} — 期望类型 {expected_type.__name__}，得到 {type(obj).__name__}")
            self._score[name] = False

    def check_raises(self, name, func, exc_type):
        """检查函数是否抛出指定异常"""
        try:
            func()
            print(f"  ❌ {name} — 没有抛出异常")
            self._score[name] = False
        except exc_type:
            print(f"  ✅ {name}")
            self._score[name] = True
        except Exception as e:
            print(f"  ❌ {name} — 抛出了 {type(e).__name__} 而非 {exc_type.__name__}")
            self._score[name] = False

    def summary(self):
        total = len(self._score)
        passed = sum(self._score.values())
        pct = (passed / total * 100) if total else 0
        bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
        print(f"\n📊 总进度: [{bar}] {passed}/{total} ({pct:.0f}%)")

C = _Checker()
print("✅ 验证工具已就绪，可以开始做题了！")

✅ 验证工具已就绪，可以开始做题了！


---
## 主题 1：推导式与生成器表达式 ⭐⭐

Python 的推导式 (comprehension) 是一种简洁强大的语法，可以用一行表达式生成列表、字典、集合，甚至惰性迭代器。

### 练习 1.1：嵌套列表推导

给定一个嵌套列表 `matrix`，请用 **一行列表推导式** 将其展平为一维列表。

```python
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
# 期望输出: [1, 2, 3, 4, 5, 6, 7, 8, 9]
```

<details><summary>💡 提示</summary>

嵌套推导的顺序与写 for 循环的顺序一致：外层循环在前，内层循环在后。
```python
[... for row in matrix for item in row]
```
</details>

In [2]:
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

# ✍️ 在此答：用一行列表推导式展开 matrix
flat = [x for row in matrix for x in row]

In [3]:
# ✅ 验证 1.1
C.check("1.1 展平嵌套列表", flat, [1, 2, 3, 4, 5, 6, 7, 8, 9])

  ✅ 1.1 展平嵌套列表


### 练习 1.2：字典推导 + 条件过滤

给定一个字典 `scores`，请用字典推导式筛选出 **分数 >= 60** 的同学，并将分数转换为等级：
- 90+ → `"A"`
- 80~89 → `"B"`
- 70~79 → `"C"`
- 60~69 → `"D"`

```python
scores = {"Alice": 95, "Bob": 42, "Carol": 78, "Dave": 63, "Eve": 88}
# 期望: {"Alice": "A", "Carol": "C", "Dave": "D", "Eve": "B"}
```

<details><summary>💡 提示</summary>

在推导式中可以调用函数或使用嵌套的三元表达式来确定等级。
</details>

In [4]:
scores = {"Alice": 95, "Bob": 42, "Carol": 78, "Dave": 63, "Eve": 88}

# ️ 在此作答
grades = {}

In [5]:
# ✅ 验证 1.2
C.check("1.2 字典推导+等级转换", grades, {"Alice": "A", "Carol": "C", "Dave": "D", "Eve": "B"})

  ❌ 1.2 字典推导+等级转换 — 你的结果: {}  期望: {'Alice': 'A', 'Carol': 'C', 'Dave': 'D', 'Eve': 'B'}


### 练习 1.3：生成器表达式 vs 列表推导

请完成以下两个任务：
1. 创建一个 **生成器表达式** `gen`，生成 0~999999 中所有能被 7 整除的数的平方
2. 用 `sum()` 对这个生成器求和，赋值给 `total`

**要求**：`gen` 必须是生成器对象（不是列表），以节省内存。

<details><summary>💡 提示</summary>

生成器表达式用圆括号 `()` 而不是方括号 `[]`。注意生成器只能迭代一次，所以先创建 gen，再立刻求和。
</details>

In [ ]:
# ✏️ 在此写代码
gen = ...  # 生成器表达式
total = ...  # 对生成器求和


In [ ]:
# ✅ 验证 1.3
C.check_type("1.3a gen是生成器", gen, types.GeneratorType)
C.check("1.3b 求和结果", total, sum(x**2 for x in range(1000000) if x % 7 == 0))

---
## 主题 2：装饰器 ⭐⭐⭐

装饰器是 Python 中修改或增强函数/类行为的强大工具。本质上是一个接收函数、返回函数的高阶函数。

### 练习 2.1：计时装饰器

编写装饰器 `timer`，功能：
- 在被装饰函数执行前后记录时间
- 将耗时（秒）存入函数的 `last_elapsed` 属性
- 正常返回原函数的返回值
- 使用 `functools.wraps` 保留原函数的元信息

<details><summary>💡 提示</summary>

```python
import functools, time
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(...)  # 调用原函数
        wrapper.last_elapsed = ...  # 记录耗时
        return result
    return wrapper
```
</details>

In [ ]:
import functools
import time

# ✏️ 在此写代码：编写 timer 装饰器
def timer(func):
    ...


@timer
def slow_add(a, b):
    """两数相加（带延迟）"""
    time.sleep(0.05)
    return a + b


In [ ]:
# ✅ 验证 2.1
result = slow_add(3, 4)
C.check("2.1a 返回值正确", result, 7)
C.check("2.1b 记录了耗时", hasattr(slow_add, 'last_elapsed') and slow_add.last_elapsed > 0.01, True)
C.check("2.1c 保留函数名", slow_add.__name__, "slow_add")
C.check("2.1d 保留文档", slow_add.__doc__, "两数相加（带延迟）")

### 练习 2.2：带参数的装饰器

编写装饰器工厂 `retry(max_attempts)`，功能：
- 当被装饰函数抛出异常时，自动重试
- 最多重试 `max_attempts` 次
- 如果所有重试都失败，抛出最后一个异常
- 如果某次调用成功，立即返回结果

<details><summary>💡 提示</summary>

带参数的装饰器需要三层嵌套：
```python
def retry(max_attempts):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(...)
                except Exception as e:
                    if attempt == max_attempts - 1:
                        raise
        return wrapper
    return decorator
```
</details>

In [ ]:
# ✏️ 在此写代码：编写 retry 装饰器工厂
def retry(max_attempts):
    ...


# 测试用：前两次调用会失败，第三次成功
_call_count_22 = 0

@retry(max_attempts=5)
def flaky_function():
    global _call_count_22
    _call_count_22 += 1
    if _call_count_22 < 3:
        raise ConnectionError(f"第 {_call_count_22} 次失败")
    return "成功"


In [ ]:
# ✅ 验证 2.2
_call_count_22 = 0
C.check("2.2a 重试后成功", lambda: flaky_function(), "成功")

@retry(max_attempts=1)
def always_fail():
    raise ValueError("永远失败")

C.check_raises("2.2b 超过次数后抛出异常", always_fail, ValueError)

---
## 主题 3：闭包与作用域 ⭐⭐⭐

闭包 (closure) 是指一个内部函数引用了外部函数作用域中的变量，即使外部函数已经返回，这些变量仍然存活。

### 练习 3.1：累加器工厂

编写函数 `make_accumulator(initial=0)`，返回一个函数 `add(n)`：
- 每次调用 `add(n)` 会将 `n` 累加到内部状态
- 返回当前累计值

```python
acc = make_accumulator(10)
acc(5)   # -> 15
acc(3)   # -> 18
acc(-8)  # -> 10
```

<details><summary>💡 提示</summary>

使用 `nonlocal` 关键字让内部函数修改外部函数的局部变量。
</details>

In [ ]:
# ✏️ 在此写代码
def make_accumulator(initial=0):
    ...


In [ ]:
# ✅ 验证 3.1
acc = make_accumulator(10)
C.check("3.1a 累加5", acc(5), 15)
C.check("3.1b 再累加3", acc(3), 18)
C.check("3.1c 累加负数", acc(-8), 10)

acc2 = make_accumulator()
C.check("3.1d 独立实例", acc2(100), 100)

### 练习 3.2：闭包陷阱

下面的代码有一个经典 bug。请修复 `make_functions` 使其正确工作。

```python
# 有 bug 的版本：
def make_functions_buggy():
    funcs = []
    for i in range(5):
        funcs.append(lambda: i)
    return funcs

# 调用后所有函数都返回 4（最后一个 i 的值）
```

修复后，`make_functions()[k]()` 应该返回 `k`。

<details><summary>💡 提示</summary>

利用默认参数在循环时「捕获」当前值：`lambda i=i: i`
</details>

In [ ]:
# ✏️ 在此写代码：修复这个函数
def make_functions():
    funcs = []
    for i in range(5):
        funcs.append(lambda: i)  # <-- 修复这一行
    return funcs


In [ ]:
# ✅ 验证 3.2
fns = make_functions()
results = [f() for f in fns]
C.check("3.2 闭包陷阱修复", results, [0, 1, 2, 3, 4])

---
## 主题 4：迭代器协议与 `yield` ⭐⭐⭐

Python 的 `for` 循环底层依赖迭代器协议（`__iter__` + `__next__`）。生成器函数 (`yield`) 是创建迭代器最优雅的方式。

### 练习 4.1：Fibonacci 生成器

编写生成器函数 `fibonacci(n)`，`yield` 前 n 个 Fibonacci 数。

```python
list(fibonacci(8))  # -> [0, 1, 1, 2, 3, 5, 8, 13]
```

<details><summary>💡 提示</summary>

```python
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b
```
</details>

In [ ]:
# ✏️ 在此写代码
def fibonacci(n):
    ...


In [ ]:
# ✅ 验证 4.1
C.check("4.1a 前8个Fibonacci", list(fibonacci(8)), [0, 1, 1, 2, 3, 5, 8, 13])
C.check("4.1b 空序列", list(fibonacci(0)), [])
C.check("4.1c 是生成器", inspect.isgeneratorfunction(fibonacci), True)

### 练习 4.2：滑动窗口生成器

编写生成器函数 `sliding_window(iterable, size)`，对任意可迭代对象产生大小为 `size` 的滑动窗口（元组）。

```python
list(sliding_window([1,2,3,4,5], 3))
# -> [(1,2,3), (2,3,4), (3,4,5)]

list(sliding_window("ABCDE", 2))
# -> [('A','B'), ('B','C'), ('C','D'), ('D','E')]
```

<details><summary>💡 提示</summary>

可以用 `collections.deque(maxlen=size)` 来维护窗口，或者先转为列表然后用索引切片。
</details>

In [ ]:
# ✏️ 在此写代码
def sliding_window(iterable, size):
    ...


In [ ]:
# ✅ 验证 4.2
C.check("4.2a 数字窗口", list(sliding_window([1,2,3,4,5], 3)), [(1,2,3),(2,3,4),(3,4,5)])
C.check("4.2b 字符串窗口", list(sliding_window("ABCDE", 2)), [('A','B'),('B','C'),('C','D'),('D','E')])
C.check("4.2c 窗口等于长度", list(sliding_window([1,2,3], 3)), [(1,2,3)])
C.check("4.2d 是生成器", inspect.isgeneratorfunction(sliding_window), True)

---
## 主题 5：魔术方法与运算符重载 ⭐⭐⭐

通过实现特殊方法 (dunder methods)，可以让自定义类支持 Python 内置运算符和函数。

### 练习 5.1：向量类

实现一个二维向量类 `Vector`，支持以下操作：

| 操作 | 示例 | 需要的魔术方法 |
|------|------|---------------|
| 创建 | `Vector(3, 4)` | `__init__` |
| 打印 | `Vector(3, 4)` → `"Vector(3, 4)"` | `__repr__` |
| 加法 | `Vector(1,2) + Vector(3,4)` → `Vector(4,6)` | `__add__` |
| 标量乘法 | `Vector(1,2) * 3` → `Vector(3,6)` | `__mul__` |
| 模长 | `abs(Vector(3,4))` → `5.0` | `__abs__` |
| 相等 | `Vector(1,2) == Vector(1,2)` → `True` | `__eq__` |
| 布尔 | `bool(Vector(0,0))` → `False` | `__bool__` |

<details><summary>💡 提示</summary>

```python
import math
class Vector:
    def __init__(self, x, y): ...
    def __repr__(self): return f"Vector({self.x}, {self.y})"
    def __add__(self, other): return Vector(self.x + other.x, ...)
    def __abs__(self): return math.sqrt(...)
```
</details>

In [ ]:
import math

# ✏️ 在此写代码
class Vector:
    ...


In [ ]:
# ✅ 验证 5.1
v1 = Vector(3, 4)
v2 = Vector(1, 2)
C.check("5.1a repr", repr(v1), "Vector(3, 4)")
C.check("5.1b 加法", v1 + v2, Vector(4, 6))
C.check("5.1c 标量乘法", v2 * 3, Vector(3, 6))
C.check("5.1d 模长", abs(v1), 5.0)
C.check("5.1e 相等", Vector(1, 2) == Vector(1, 2), True)
C.check("5.1f 不等", Vector(1, 2) == Vector(1, 3), False)
C.check("5.1g 零向量为假", bool(Vector(0, 0)), False)
C.check("5.1h 非零向量为真", bool(v1), True)

---
## 主题 6：上下文管理器 ⭐⭐⭐

上下文管理器通过 `with` 语句自动管理资源的获取和释放（如文件、锁、数据库连接）。

### 练习 6.1：用 `__enter__` / `__exit__` 实现

编写一个上下文管理器类 `Indenter`，实现自动缩进打印：

```python
with Indenter() as indent:
    indent.print("第一层")        # 输出: "  第一层"
    with indent:
        indent.print("第二层")    # 输出: "    第二层"
    indent.print("回到第一层")    # 输出: "  回到第一层"
```

每进入一层 `with`，缩进增加 2 个空格；退出时减少。

<details><summary>💡 提示</summary>

```python
class Indenter:
    def __init__(self):
        self._level = 0
    def __enter__(self):
        self._level += 1
        return self
    def __exit__(self, *args):
        self._level -= 1
    def print(self, text):
        print("  " * self._level + text)
```
</details>

In [ ]:
# ✏️ 在此写代码
class Indenter:
    ...


In [ ]:
# ✅ 验证 6.1
import io, sys
buf = io.StringIO()
_old = sys.stdout
sys.stdout = buf
with Indenter() as indent:
    indent.print("a")
    with indent:
        indent.print("b")
        with indent:
            indent.print("c")
        indent.print("d")
    indent.print("e")
sys.stdout = _old
lines = buf.getvalue().strip("\n").split("\n")
C.check("6.1a 第一层缩进", lines[0], "  a")
C.check("6.1b 第二层缩进", lines[1], "    b")
C.check("6.1c 第三层缩进", lines[2], "      c")
C.check("6.1d 退回第二层", lines[3], "    d")
C.check("6.1e 退回第一层", lines[4], "  e")

### 练习 6.2：用 `contextlib` 实现

使用 `contextlib.contextmanager` 装饰器编写一个上下文管理器 `suppress(*exceptions)`，功能：
- 捕获并静默忽略指定类型的异常
- 不在指定范围内的异常正常抛出

```python
with suppress(ValueError, TypeError):
    int("abc")  # ValueError 被忽略
print("继续执行")  # 正常到达
```

<details><summary>💡 提示</summary>

```python
from contextlib import contextmanager

@contextmanager
def suppress(*exceptions):
    try:
        yield
    except exceptions:
        pass
```
</details>

In [ ]:
from contextlib import contextmanager

# ✏️ 在此写代码
@contextmanager
def suppress(*exceptions):
    ...


In [ ]:
# ✅ 验证 6.2
reached = False
with suppress(ValueError):
    int("abc")
reached = True
C.check("6.2a 忽略ValueError", reached, True)

def _test_62b():
    with suppress(ValueError):
        raise KeyError('test')
C.check_raises("6.2b 不忽略KeyError", _test_62b, KeyError)

---
## 主题 7：`*args` / `**kwargs` 与高级解包 ⭐⭐

### 练习 7.1：合并字典函数

编写函数 `deep_merge(*dicts)`，将多个字典深度合并。后面的字典优先级更高。

规则：
- 如果同一个 key 在多个字典中都存在且值都是 dict，则递归合并
- 否则后面的值覆盖前面的值

```python
deep_merge(
    {"a": 1, "b": {"x": 10, "y": 20}},
    {"b": {"y": 99, "z": 30}, "c": 3}
)
# -> {"a": 1, "b": {"x": 10, "y": 99, "z": 30}, "c": 3}
```

<details><summary>💡 提示</summary>

先初始化一个空 result 字典，然后逐个遍历输入字典的键值对。如果 key 已存在于 result 中且两边都是 dict，就递归 `deep_merge`；否则直接覆盖。
</details>

In [ ]:
# ✏️ 在此写代码
def deep_merge(*dicts):
    ...


In [ ]:
# ✅ 验证 7.1
C.check("7.1a 深度合并", deep_merge(
    {"a": 1, "b": {"x": 10, "y": 20}},
    {"b": {"y": 99, "z": 30}, "c": 3}
), {"a": 1, "b": {"x": 10, "y": 99, "z": 30}, "c": 3})

C.check("7.1b 三个字典", deep_merge({"a": 1}, {"b": 2}, {"a": 10, "c": 3}), {"a": 10, "b": 2, "c": 3})
C.check("7.1c 空输入", deep_merge(), {})

### 练习 7.2：函数签名转换

编写函数 `call_with_matching_args(func, **all_kwargs)`：
- 检查 `func` 的参数签名
- 只传入 `func` 接受的参数，忽略多余的
- 返回 `func` 的返回值

```python
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

call_with_matching_args(greet, name="Alice", greeting="Hi", age=25, city="NYC")
# -> "Hi, Alice!"  (age 和 city 被忽略)
```

<details><summary>💡 提示</summary>

使用 `inspect.signature(func).parameters` 获取函数接受的参数名，然后从 `all_kwargs` 中过滤出对应的键值对。注意要处理 `**kwargs` 类型的参数（`VAR_KEYWORD`）——如果函数接受 `**kwargs`，就不需要过滤。
</details>

In [ ]:
import inspect

# ✏️ 在此写代码
def call_with_matching_args(func, **all_kwargs):
    ...


In [ ]:
# ✅ 验证 7.2
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

C.check("7.2a 过滤多余参数",
    call_with_matching_args(greet, name="Alice", greeting="Hi", age=25), "Hi, Alice!")
C.check("7.2b 使用默认值",
    call_with_matching_args(greet, name="Bob", city="NYC"), "Hello, Bob!")

def accept_all(**kwargs):
    return kwargs
C.check("7.2c 函数接受**kwargs",
    call_with_matching_args(accept_all, x=1, y=2), {"x": 1, "y": 2})

---
## 主题 8：函数式编程 ⭐⭐⭐

Python 支持 `map`、`filter`、`reduce`、`lambda` 等函数式编程工具，同时 `functools` 模块提供了许多高阶函数。

### 练习 8.1：管道 (Pipe) 函数

编写函数 `pipe(value, *functions)`，将 `value` 依次传入每个函数，返回最终结果。

```python
pipe("  Hello, World!  ",
     str.strip,
     str.lower,
     lambda s: s.replace(",", "")
)
# -> "hello world!"
```

<details><summary>💡 提示</summary>

可以使用 `functools.reduce`，也可以用简单的 for 循环。
</details>

In [ ]:
from functools import reduce

# ✏️ 在此写代码
def pipe(value, *functions):
    ...


In [ ]:
# ✅ 验证 8.1
C.check("8.1a 字符串管道", pipe(
    "  Hello, World!  ", str.strip, str.lower, lambda s: s.replace(",", "")
), "hello world!")

C.check("8.1b 数字管道", pipe(2, lambda x: x+3, lambda x: x*10, str), "50")
C.check("8.1c 无函数", pipe(42), 42)

### 练习 8.2：缓存装饰器 (`lru_cache` 手动实现)

不使用 `functools.lru_cache`，手动实现一个 `memoize` 装饰器：
- 缓存函数的调用结果
- 相同参数第二次调用时直接返回缓存值
- 被装饰函数增加 `cache_info()` 方法，返回 `{"hits": N, "misses": M}`

<details><summary>💡 提示</summary>

用字典存储 `args -> result` 的映射。注意 `args` 需要是可哈希的（元组）。
</details>

In [ ]:
# ✏️ 在此写代码
def memoize(func):
    ...


In [ ]:
# ✅ 验证 8.2
_fib_calls = 0

@memoize
def fib(n):
    global _fib_calls
    _fib_calls += 1
    if n < 2: return n
    return fib(n-1) + fib(n-2)

_fib_calls = 0
C.check("8.2a fib(10)结果", fib(10), 55)
C.check("8.2b 缓存生效(调用次数少)", _fib_calls <= 11, True)  # 无缓存需要 177 次

info = fib.cache_info()
C.check("8.2c cache_info存在", "hits" in info and "misses" in info, True)
C.check("8.2d 有缓存命中", info["hits"] > 0, True)

---
## 主题 9：Dataclass 与类型提示 ⭐⭐

`dataclasses` 模块可以自动为类生成 `__init__`、`__repr__`、`__eq__` 等方法。结合类型提示可以写出简洁又清晰的数据类。

### 练习 9.1：学生成绩单

使用 `@dataclass` 创建 `Student` 类：

- 字段：`name: str`、`scores: list[int]`（使用 `field(default_factory=list)`）
- 属性 `average`（用 `@property`）返回平均分（保留 1 位小数），空列表返回 `0.0`
- 属性 `grade` 返回等级（同主题1的分级规则：A/B/C/D/F）
- 实现 `__lt__` 使 Student 可以按平均分排序（平均分低的「小」）

<details><summary>💡 提示</summary>

```python
from dataclasses import dataclass, field

@dataclass
class Student:
    name: str
    scores: list[int] = field(default_factory=list)

    @property
    def average(self) -> float:
        return round(sum(self.scores) / len(self.scores), 1) if self.scores else 0.0
```
</details>

In [ ]:
from dataclasses import dataclass, field

# ✏️ 在此写代码
@dataclass
class Student:
    ...


In [ ]:
# ✅ 验证 9.1
s1 = Student("Alice", [95, 88, 92])
s2 = Student("Bob", [60, 55, 70])
s3 = Student("Carol")

C.check("9.1a 平均分", s1.average, 91.7)
C.check("9.1b 等级A", s1.grade, "A")
C.check("9.1c 等级D", s2.grade, "D")
C.check("9.1d 空成绩", s3.average, 0.0)
C.check("9.1e 空成绩等级", s3.grade, "F")
C.check("9.1f 排序", sorted([s1, s2, s3]), [s3, s2, s1])
C.check("9.1g 自动repr", "Alice" in repr(s1), True)

---
## 主题 10：异步编程 async/await ⭐⭐⭐⭐

`async/await` 是 Python 处理 I/O 密集型并发任务的方式。协程可以在等待 I/O 时让出执行权，让其他协程运行。

### 练习 10.1：并发执行

编写异步函数 `fetch_all(urls)`：
- 接收一个 URL 列表
- 对每个 URL，调用提供的 `fake_fetch(url)` 获取结果
- 使用 `asyncio.gather` **并发**执行所有请求
- 返回结果列表（顺序与输入 URL 一致）

我们提供一个模拟的异步函数 `fake_fetch`，每次调用需要 0.1 秒。如果你正确使用了并发，3 个请求总耗时应该约 0.1 秒而非 0.3 秒。

<details><summary>💡 提示</summary>

```python
async def fetch_all(urls):
    tasks = [fake_fetch(url) for url in urls]
    return await asyncio.gather(*tasks)
```
</details>

In [ ]:
import asyncio

async def fake_fetch(url):
    """模拟网络请求，0.1秒后返回结果"""
    await asyncio.sleep(0.1)
    return f"Response from {url}"

# ✏️ 在此写代码
async def fetch_all(urls):
    ...


In [ ]:
# ✅ 验证 10.1
import time as _time

async def _test_fetch():
    urls = ["https://a.com", "https://b.com", "https://c.com"]
    start = _time.perf_counter()
    results = await fetch_all(urls)
    elapsed = _time.perf_counter() - start
    return results, elapsed

_results, _elapsed = await _test_fetch()
C.check("10.1a 返回结果数量", len(_results), 3)
C.check("10.1b 结果顺序正确", _results[0], "Response from https://a.com")
C.check("10.1c 并发执行(耗时<0.5s)", _elapsed < 0.5, True)

### 练习 10.2：异步生成器

编写异步生成器 `async_countdown(n)`：
- 从 `n` 倒数到 `1`
- 每次 yield 之间等待 0.05 秒（模拟异步操作）
- 最后 yield `"发射！"`

```python
async for item in async_countdown(3):
    print(item)
# 输出: 3, 2, 1, "发射！"
```

<details><summary>💡 提示</summary>

```python
async def async_countdown(n):
    for i in range(n, 0, -1):
        await asyncio.sleep(0.05)
        yield i
    yield "发射！"
```
</details>

In [ ]:
# ✏️ 在此写代码
async def async_countdown(n):
    ...


In [ ]:
# ✅ 验证 10.2
async def _test_countdown():
    items = []
    async for item in async_countdown(3):
        items.append(item)
    return items

_cd_result = await _test_countdown()
C.check("10.2a 倒数结果", _cd_result, [3, 2, 1, "发射！"])

---
## 📊 查看总进度

运行下面的单元格查看你完成了多少题！

In [ ]:
C.summary()

---

## 🎉 完成所有练习？

恭喜！你已经掌握了 Python 进阶的核心知识。接下来可以：

- 尝试不看提示独立完成每道题
- 修改题目参数，验证你的代码在边界条件下也能正确工作
- 探索更高阶主题：元类 (metaclass)、描述符 (descriptor)、`__slots__`、ABC 抽象基类等
- 查看本课程的其他 Lesson，学习如何将 Python 应用于 AI Agent 开发